In [1]:
import os
import pandas as pd

In [2]:
BASE_DIR = "./trainings"
RESULTS_DIR = "./results"

OUTPUT_EXCEL = os.path.join(
    RESULTS_DIR,
    "Confusion_Matrixes.xlsx"
)

GROUP_CONFIG = {
    "01": {"states": 4,  "gaussians": 9},
    "02": {"states": 3,  "gaussians": 9},
    "03": {"states": 12, "gaussians": 14},
    "04": {"states": 3,  "gaussians": 10},
    "05": {"states": 5,  "gaussians": 10},
    "06": {"states": 6,  "gaussians": 1},
    "07": {"states": 4,  "gaussians": 7},
    "08": {"states": 4,  "gaussians": 13},
    "09": {"states": 5,  "gaussians": 10},
    "10": {"states": 4,  "gaussians": 8},
}

CLASS_ORDER = [
    "SegundoPRIMARIA",
    "QuintoPRIMARIA",
    "SegundoESO"
]

In [3]:
def read_reference_mlf(path):
    labels = {}

    with open(path, "r") as f:
        lines = [line.strip() for line in f]

    i = 0
    while i < len(lines):
        if lines[i].startswith('"'):
            filename = os.path.basename(lines[i].replace('"', ''))
            sample_id = filename.replace(".lab", "")
            labels[sample_id] = lines[i + 1]
            i += 3
        else:
            i += 1

    return labels

In [4]:
def read_recout_mlf(path):
    labels = {}

    with open(path, "r") as f:
        lines = [line.strip() for line in f]

    i = 0
    while i < len(lines):
        if lines[i].startswith('"'):
            filename = os.path.basename(lines[i].replace('"', ''))
            sample_id = filename.replace(".rec", "")

            prediction_line = lines[i + 1]
            predicted_label = prediction_line.split()[2]

            labels[sample_id] = predicted_label
            i += 3
        else:
            i += 1

    return labels

In [5]:
def add_metrics(matrix):

    result = matrix.copy()

    precisions = {}

    for cls in matrix.index:
        tp = matrix.loc[cls, cls]
        total_pred = matrix[cls].sum()
        precisions[cls] = (tp / total_pred) if total_pred > 0 else 0

    result.loc["Precision"] = precisions

    sensitivities = {}

    for cls in matrix.columns:
        tp = matrix.loc[cls, cls]
        total_real = matrix.loc[cls].sum()
        sensitivities[cls] = (tp / total_real) if total_real > 0 else 0

    result["Sensitivity"] = pd.Series(sensitivities)

    return result

In [6]:
all_classes = CLASS_ORDER

general_matrix = pd.DataFrame(
    0,
    index=all_classes,
    columns=all_classes
)

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:

    for group, cfg in GROUP_CONFIG.items():

        print(f"\nProcessing Group {group}")

        test_mlf = os.path.join(
            BASE_DIR,
            f"Train{group}",
            f"test{group}.mlf"
        )

        recout_mlf = os.path.join(
            RESULTS_DIR,
            f"results_{cfg['states']}_states",
            f"Group_{group}",
            f"{cfg['gaussians']}_gaussians",
            "HVite",
            f"recout{group}.mlf"
        )

        if not os.path.exists(test_mlf):
            print(f"Missing: {test_mlf}")
            continue

        if not os.path.exists(recout_mlf):
            print(f"Missing: {recout_mlf}")
            continue

        true_labels = read_reference_mlf(test_mlf)
        predicted_labels = read_recout_mlf(recout_mlf)

        matrix = pd.DataFrame(
            0,
            index=all_classes,
            columns=all_classes
        )

        for sample_id, true_class in true_labels.items():

            if sample_id not in predicted_labels:
                continue

            predicted_class = predicted_labels[sample_id]

            matrix.loc[true_class, predicted_class] += 1
            general_matrix.loc[true_class, predicted_class] += 1

        matrix_with_metrics = add_metrics(matrix)

        sheet_name = f"Group_{group}"
        matrix_with_metrics.to_excel(writer, sheet_name=sheet_name)

        ws = writer.book[sheet_name]

        max_row = ws.max_row
        max_col = ws.max_column

        for row in range(1, max_row + 1):
            if ws.cell(row=row, column=1).value == "Precision":

                for col in range(2, max_col + 1):
                    ws.cell(row=row, column=col).number_format = '0.00%'

                break

        for col in range(1, max_col + 1):
            if ws.cell(row=1, column=col).value == "Sensitivity":

                for row in range(2, max_row + 1):
                    ws.cell(row=row, column=col).number_format = '0.00%'

                break

    general_with_metrics = add_metrics(general_matrix)

    general_with_metrics.to_excel(writer, sheet_name="General")

    ws = writer.book["General"]

    max_row = ws.max_row
    max_col = ws.max_column

    for row in range(1, max_row + 1):
        if ws.cell(row=row, column=1).value == "Precision":

            for col in range(2, max_col + 1):
                ws.cell(row=row, column=col).number_format = '0.00%'

            break

    for col in range(1, max_col + 1):
        if ws.cell(row=1, column=col).value == "Sensitivity":

            for row in range(2, max_row + 1):
                ws.cell(row=row, column=col).number_format = '0.00%'

            break

print(f"Excel generated correctly:{OUTPUT_EXCEL}")


Processing Group 01

Processing Group 02

Processing Group 03

Processing Group 04

Processing Group 05

Processing Group 06

Processing Group 07

Processing Group 08

Processing Group 09

Processing Group 10
Excel generated correctly:./results\Confusion_Matrixes.xlsx
